# 04 · Encoding: sentence embeddings

**Workflow step: Step 2 (b) of Figure 1 in the proposal** — the second encoding. It is numbered after notebook 03 because the first baselines were run on the one-hot encoding before the embeddings were built.

**Where we are.** Notebook 01 built one row per entry and the JET flags (`meta.parquet`). Notebook 02 built the one-hot encoding: per line slot an account block (25 accounts + PAD) and a side block (D / C / PAD), i.e. 116 of the 131 main columns describe *which accounts on which side*; the rest are amounts, user and time. In one-hot every account is equally different from every other one, and the block grows with accounts × slots (500 accounts and 10 slots would give 5,010 columns). Notebook 03 ran the first baselines on this encoding: LOF reached AP 0.144 on FY2024, Isolation Forest 0.061, JET-only 0.092.

## Purpose of this notebook
RQ2 asks whether a pre-trained sentence embedding represents the entries better than one-hot. This notebook builds that second encoding. Following D16, the embedding replaces **only the account + side block**; all other columns (slot amounts, user, time) are taken unchanged from notebook 02, so RQ2 compares the representation of the accounts and nothing else. Each entry becomes a short sentence of its lines (account names and the words Debit / Credit), and the sentence is embedded with all-MiniLM-L6-v2. Combining the embedding with the other columns and weighting the two blocks (D17), and running the detectors on it, follow in notebook 05.

Where each goal is met:
1. **Load** the posting lines and the meta table; check that account names identify the accounts — section 1.
2. **Build the sentences** in two templates — `tc` (main) and `bak` (exact format of Bakumenko et al., 2025, as ablation) — with the lines in the D9 order of notebook 02 (D16) — section 2.
3. **Describe the booking patterns** the sentences stand for, and which anomaly types create new patterns — section 3.
4. **Embed** the distinct sentences (L2-normalised) — section 4.
5. **First look at RQ2**: distances between the patterns that matter, one-hot vs embedding — section 5.
6. **Save** the unique embeddings and the per-entry index — section 6.
7. **How later notebooks use it** — section 7.

**Needs:** `pip install sentence-transformers` (pulls in torch); the model (~90 MB) is downloaded once. Runs in seconds on a laptop CPU, because the whole ledger has only 29 distinct sentences.

<div dir="rtl">

## هدف این notebook (فارسی)

**مرحله در workflow: ‏Step 2 (b) در Figure 1 پروپوزال** — encoding دوم. شماره‌اش بعد از 03 است چون baselineهای اول پیش از ساخت embeddingها روی one-hot اجرا شدند.

**تا اینجا چه گذشت.** notebook 01 برای هر سند یک ردیف و پرچم‌های JET ساخت (`meta.parquet`). notebook 02 encoding ‏one-hot را ساخت: در هر slot خط یک بلوک حساب (25 حساب + PAD) و یک بلوک طرف (D / C / PAD)؛ یعنی 116 ستون از 131 ستون تنظیم اصلی می‌گویند *کدام حساب‌ها در کدام طرف*؛ بقیه مبلغ، کاربر و زمان‌اند. در one-hot هر حساب با هر حساب دیگر به یک اندازه متفاوت است و بلوک با «تعداد حساب × تعداد slot» بزرگ می‌شود (500 حساب و 10 slot یعنی 5,010 ستون). notebook 03 اولین baselineها را روی این encoding اجرا کرد: ‏LOF روی FY2024 به AP ‏0.144 رسید، Isolation Forest به 0.061 و JET-only به 0.092.

**هدف.** RQ2 می‌پرسد آیا sentence embedding از پیش آموزش‌دیده سندها را بهتر از one-hot نمایش می‌دهد. این notebook آن encoding دوم را می‌سازد. طبق D16، embedding **فقط بلوک حساب + طرف** را جایگزین می‌کند؛ بقیهٔ ستون‌ها (مبلغ slotها، کاربر، زمان) بدون تغییر از notebook 02 می‌آیند، تا RQ2 فقط نمایش حساب‌ها را مقایسه کند. هر سند به یک جملهٔ کوتاه از خطوطش تبدیل می‌شود (نام حساب‌ها و کلمه‌های Debit / Credit) و جمله با all-MiniLM-L6-v2 جاسازی می‌شود. ترکیب embedding با بقیهٔ ستون‌ها و وزن‌دهی دو بلوک (D17) و اجرای detectorها روی آن در notebook 05 می‌آیند.

این هدف‌ها کجا محقق شده‌اند:
۱. بارگذاری خطوط و جدول meta؛ چک اینکه نام حساب‌ها حساب‌ها را یکتا مشخص می‌کنند – بخش ۱
۲. ساخت جمله‌ها در دو قالب — `tc` (اصلی) و `bak` (قالب دقیق Bakumenko et al., 2025، برای ablation) — با ترتیب خطوط D9 از notebook 02 ‏(D16) – بخش ۲
۳. توصیف الگوهای ثبتی که جمله‌ها نمایندهٔ آن‌اند و اینکه کدام نوع ناهنجاری الگوی تازه می‌سازد – بخش ۳
۴. جاسازی جمله‌های متمایز (نرمال‌شده با L2) – بخش ۴
۵. اولین نگاه به RQ2: فاصلهٔ الگوهای مهم، one-hot در برابر embedding – بخش ۵
۶. ذخیرهٔ embeddingهای یکتا و اندیس هر سند – بخش ۶
۷. نحوهٔ استفاده در notebookهای بعدی – بخش ۷

**نیاز:** ‏`pip install sentence-transformers` (که torch را هم نصب می‌کند)؛ مدل (حدود 90 MB) یک بار دانلود می‌شود. روی CPU لپ‌تاپ چند ثانیه طول می‌کشد چون کل دفتر فقط 29 جملهٔ متمایز دارد.

</div>

## Inputs, outputs and decisions used

**Input:** `data/raw/journal_entries_4203.csv`, `data/derived/meta.parquet` (notebook 01), `data/derived/X_onehot.npz` + `onehot_features.json` (notebook 02; used in sections 5 and 7).
**Output (per template):** `data/derived/emb_<template>.npy` (unique sentences × 384), `emb_<template>_index.npy` (one sentence id per entry, `meta.parquet` order), `emb_<template>.json` (sentences, counts, model and library versions).

| Template | Sentence (entry `a_1`) | Role |
|---|---|---|
| `tc` (T-C) | `Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Trade payables.` | main setting: embedding + the non-account one-hot columns |
| `bak` (T-Bak) | `Account_DC: 1576_D, Account_DC: 3400_D, Account_DC: 1600_C` | ablation: exact Bakumenko et al. (2025) format with codes |

| ID | What | Setting | Section |
|---|---|---|---|
| D16 | what goes into the sentence | lines in D9 order; account **name** instead of code; the words Debit / Credit; `; ` as separator (names contain commas); nothing else (amounts, user and time stay one-hot columns) | 2 |
| D1 | free text `text` | not in the sentence (as in notebook 02) | 2 |
| D17 | weight of the embedding block vs the other columns | decided in notebook 05; this notebook stores the raw, L2-normalised embeddings only | 6, 7 |

T-A (text only) uses the `tc` embedding without the other columns, so it needs no separate file.

In [1]:
# cell_04_01
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)
RAW_REL = Path("data/raw/journal_entries_4203.csv")


def find_project_root(marker: Path = RAW_REL) -> Path:
    # walk up from the working directory until `marker` exists (same as notebooks 01-03)
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found from {here} upwards")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))          # makes src/ importable (laptop and Colab)
from src.load import load_lines
from src import embed, encode as fe

DATA = ROOT / RAW_REL
DERIVED = ROOT / "data" / "derived"
print("project root:", ROOT, "| model:", embed.MODEL_NAME)

project root: C:\MyWorks\Thesis\Jet | model: sentence-transformers/all-MiniLM-L6-v2


## 1. Load the posting lines and the meta table

**Goal.** Read the lines and `meta.parquet`, whose row order is the order of every matrix in the project. The sentences use account **names**, so each account must have exactly one name — otherwise two accounts could get the same words, or one account two different sentences.


<div dir="rtl">

**هدف.** خطوط و `meta.parquet` خوانده می‌شوند؛ ترتیب سطرهای meta ترتیب همهٔ ماتریس‌های پروژه است. جمله‌ها از **نام** حساب استفاده می‌کنند، پس هر حساب باید دقیقاً یک نام داشته باشد؛ وگرنه دو حساب ممکن بود کلمات یکسان بگیرند یا یک حساب دو جملهٔ متفاوت.

</div>

In [2]:
# cell_04_02
lines = load_lines(DATA)
meta = pd.read_parquet(DERIVED / "meta.parquet")
print("posting lines:", lines.shape, "| entries:", meta.shape)
assert set(lines["posting_id"]) == set(meta["posting_id"])
names_per_account = lines.groupby("gl_account")["gl_account_name"].nunique()
accounts_per_name = lines.groupby("gl_account_name")["gl_account"].nunique()
assert names_per_account.max() == 1 and accounts_per_name.max() == 1      # one-to-one
print("accounts:", lines["gl_account"].nunique(), "| names:", lines["gl_account_name"].nunique())

posting lines: (256232, 21) | entries: (102429, 11)
accounts: 25 | names: 25


**Result.** 256,232 lines, 102,429 entries; 25 accounts with 25 names, one-to-one. Replacing the code by the name loses no information.


<div dir="rtl">

**نتیجه.** ‏256,232 خط و 102,429 سند؛ 25 حساب با 25 نام، یک‌به‌یک. جایگزینی کد با نام هیچ اطلاعاتی از دست نمی‌دهد.

</div>

## 2. Build the sentences (D16)

**Goal.** One sentence per entry, in two templates. The lines are put in the **D9 order** of notebook 02 (debit first, then account number, then amount), so that both encodings see the lines in the same order. `tc` writes the side as a word and the account by its name, separated by `; ` because names contain commas; `bak` is the exact format of Bakumenko et al. (2025) with codes. Amounts, user and time are **not** written into the sentence: they stay as the one-hot columns of notebook 02, so that RQ2 compares only the account representation (D16; numbers are also poorly represented by such models — Wallace et al., 2019). Only the distinct sentences are embedded; an index maps every entry (meta order) to its sentence.


<div dir="rtl">

**هدف.** برای هر سند یک جمله، در دو قالب. خطوط به **ترتیب D9** notebook 02 چیده می‌شوند (اول بدهکار، بعد شمارهٔ حساب، بعد مبلغ) تا هر دو encoding خطوط را به یک ترتیب ببینند. `tc` طرف را با کلمه و حساب را با نامش می‌نویسد و با `; ` جدا می‌کند چون نام حساب‌ها ویرگول دارند؛ `bak` قالب دقیق Bakumenko et al. (2025) با کد حساب است. مبلغ، کاربر و زمان در جمله **نوشته نمی‌شوند**: همان ستون‌های one-hot نوت‌بوک 02 می‌مانند تا RQ2 فقط نمایش حساب‌ها را مقایسه کند (D16؛ این مدل‌ها عدد را هم خوب نمایش نمی‌دهند — Wallace et al., 2019). فقط جمله‌های متمایز جاسازی می‌شوند و یک اندیس هر سند (به ترتیب meta) را به جمله‌اش وصل می‌کند.

</div>

In [3]:
# cell_04_03
sentences, texts, index = {}, {}, {}
for t in embed.TEMPLATES:
    sentences[t] = embed.build_sentences(lines, meta, t)          # meta order, lines in D9 order
    texts[t], index[t] = embed.unique_index(sentences[t])           # distinct sentences + index per entry
    assert sentences[t].notna().all() and len(sentences[t]) == len(meta)
    assert (np.array(texts[t])[index[t]] == sentences[t].to_numpy()).all()   # the index restores every sentence
    print(f"{t}: {len(texts[t])} distinct sentences for {len(meta):,} entries")
i_a1 = int(np.flatnonzero(meta["posting_id"].to_numpy() == "a_1")[0])
for t in embed.TEMPLATES:
    print(f"  a_1 [{t}]: {sentences[t].iloc[i_a1]}")

tc: 29 distinct sentences for 102,429 entries
bak: 29 distinct sentences for 102,429 entries
  a_1 [tc]: Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Trade payables.
  a_1 [bak]: Account_DC: 1576_D, Account_DC: 3400_D, Account_DC: 1600_C


**Result.** Both templates give **29 distinct sentences** for 102,429 entries — one per booking pattern found in notebook 02 (section 3). The embedding is therefore a semantic embedding of the booking pattern, not of the individual entry; everything that varies between entries of the same pattern (amounts, user, time) comes from the one-hot columns.


<div dir="rtl">

**نتیجه.** هر دو قالب برای 102,429 سند **29 جملهٔ متمایز** می‌دهند — یکی برای هر الگوی ثبتی که notebook 02 (بخش ۳) پیدا کرد. پس embedding در واقع جاسازی معنایی الگوی ثبت است، نه تک‌تک سندها؛ هر چه بین سندهای یک الگو فرق می‌کند (مبلغ، کاربر، زمان) از ستون‌های one-hot می‌آید.

</div>

## 3. The booking patterns behind the sentences

**Goal.** Show what the 29 sentences stand for: entries, anomalies and anomaly types per pattern. Labels are used only to describe the dataset (both years, as in notebook 01); no choice is made from this table. The question for RQ2: which anomaly types create patterns that normal entries do not have — the only place where the account representation can matter directly?


<div dir="rtl">

**هدف.** نشان دادن اینکه 29 جمله نمایندهٔ چه هستند: تعداد سند، ناهنجاری و نوع ناهنجاری در هر الگو. برچسب فقط برای توصیف دیتاست به کار می‌رود (هر دو سال، مثل notebook 01) و هیچ انتخابی از این جدول نمی‌آید. پرسش برای RQ2: کدام نوع ناهنجاری الگویی می‌سازد که سندهای عادی ندارند — تنها جایی که نمایش حساب‌ها می‌تواند مستقیماً اثر داشته باشد؟

</div>

In [4]:
# cell_04_04
desc = (pd.DataFrame({"sentence": sentences["tc"], "label": meta["label"].to_numpy(),
                      "marking": meta["marking"].to_numpy()})
        .groupby("sentence", sort=False)
        .agg(entries=("label", "size"), anomalies=("label", "sum"),
             types=("marking", lambda s: ",".join(f"M{m}" for m in sorted(set(s)) if m > 0)))
        .sort_values("entries", ascending=False))
only_anomalies = desc[desc["entries"] == desc["anomalies"]]
print(f"patterns with anomalies only: {len(only_anomalies)} ({only_anomalies['entries'].sum()} entries), "
      f"types: {sorted(set(','.join(only_anomalies['types']).split(',')))}")
desc

patterns with anomalies only: 6 (118 entries), types: ['M1', 'M7']


,entries,anomalies,types
sentence,,,
"Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Trade payables.",32944,159,"M3,M4,M5"
Debit Trade payables; Credit Bank (Postbank).,21796,248,"M3,M5,M6,M8"
"Debit Trade receivables; Credit VAT, 19%; Credit Revenue, 19%.",18000,91,"M3,M4,M5"
Debit Bank (Postbank); Credit Trade receivables.,16956,56,"M3,M5,M6,M8"
Debit Trade payables; Credit Cash-in-hand.,10985,172,"M1,M2,M3,M5,M6,M8"
Debit Cash-in-hand; Credit Trade receivables.,993,6,"M3,M5"
Debit Salaries; Credit Bank (Postbank).,239,0,
Debit Wages and salaries; Debit Statutory social security expenses; Credit Bank (Postbank).,119,0,
"Debit Trade payables; Credit Deductible input tax, 19%; Credit Cost of merchandise, 19% input tax.",60,0,


**Result.**
- Five patterns hold 98 % of the entries (purchases, payments, sales, receipts, cash payments); 23 of the 29 patterns have fewer than 250 entries.
- **Six patterns contain anomalies only** (118 entries): the payment patterns via account 1200 "Bank" (all 60 M7 entries) and the purchase / rent patterns paid from cash-in-hand (58 of the 120 M1 entries). These are the anomalies that **change the accounts**; all other types (M2–M6, M8, and the rest of M1) sit inside frequent normal patterns and differ only in amounts, user or time.
- So the account representation can matter directly for at most these 118 entries (14 % of the anomalies); for the others the embedding can only help by placing patterns differently relative to each other.


<div dir="rtl">

**نتیجه.**
- پنج الگو 98٪ سندها را دارند (خرید، پرداخت، فروش، دریافت، پرداخت نقدی)؛ 23 الگو از 29 الگو کمتر از 250 سند دارند.
- **شش الگو فقط ناهنجاری دارند** (118 سند): الگوهای پرداخت از حساب 1200 «Bank» (هر 60 سند M7) و الگوهای خرید/اجاره با پرداخت از صندوق (58 از 120 سند M1). این‌ها ناهنجاری‌هایی‌اند که **حساب‌ها را عوض می‌کنند**؛ بقیهٔ انواع (M2 تا M6، M8 و بقیهٔ M1) داخل الگوهای پرتکرار عادی‌اند و فقط در مبلغ، کاربر یا زمان فرق دارند.
- پس نمایش حساب‌ها حداکثر برای همین 118 سند (14٪ ناهنجاری‌ها) مستقیماً اثر دارد؛ برای بقیه، embedding فقط با جای متفاوتی که به الگوها نسبت به هم می‌دهد می‌تواند کمک کند.

</div>

## 4. Embed the distinct sentences

**Goal.** Embed the 29 distinct sentences of each template with all-MiniLM-L6-v2 (384 dimensions), L2-normalised, so that the dot product of two vectors is their cosine similarity and their Euclidean distance is √(2 − 2·cos).


<div dir="rtl">

**هدف.** ‏29 جملهٔ متمایز هر قالب با all-MiniLM-L6-v2 (‏384 بعد) جاسازی و با L2 نرمال می‌شوند، تا ضرب داخلی دو بردار همان شباهت کسینوسی باشد و فاصلهٔ اقلیدسی‌شان √(2 − 2·cos).

</div>

In [5]:
# cell_04_05
emb = {}
for t in embed.TEMPLATES:
    emb[t] = embed.encode(texts[t])
    assert emb[t].shape == (len(texts[t]), 384)
    assert np.allclose(np.linalg.norm(emb[t], axis=1), 1.0, atol=1e-4)
    print(f"{t}: {emb[t].shape}")

c:\MyWorks\Thesis\Jet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2942.79it/s]


tc: (29, 384)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2640.91it/s]


bak: (29, 384)


**Result.** Two matrices of 29 × 384, all rows of length 1. (Run of 26 Nov: about 12 s for the first template including loading the model, under 1 s for the second.)


<div dir="rtl">

**نتیجه.** دو ماتریس 29 × 384 که طول همهٔ سطرهایشان 1 است. (اجرای 26 نوامبر: حدود 12 ثانیه برای قالب اول با بارگذاری مدل، کمتر از 1 ثانیه برای قالب دوم.)

</div>

## 5. First look at RQ2: distances between the patterns that matter

**Goal.** Compare how far apart the two encodings place three pairs of patterns: (a) an M7 payment via "Bank" (1200) vs the normal payment via "Bank (Postbank)"; (b) an M1 purchase paid from cash-in-hand vs the normal purchase on account; (c) as a reference, a normal payment vs a normal receipt (accounts and sides swapped). For one-hot the distance is taken in the account + side block of notebook 02; for the embeddings it is the Euclidean distance √(2 − 2·cos). Hypothesis (D16): one-hot puts (a) and (b) at the same distance, because one account is replaced in one slot in both; the names make (a) much closer than (b), because "Bank" and "Bank (Postbank)" read almost the same — which could hide M7 from a distance-based detector. This is a description of the representation only; the detectors in notebook 05 decide.


<div dir="rtl">

**هدف.** مقایسهٔ اینکه دو encoding سه جفت الگو را چقدر از هم دور می‌گذارند: (الف) پرداخت M7 از «Bank» (1200) در برابر پرداخت عادی از «Bank (Postbank)»؛ (ب) خرید M1 با پرداخت از صندوق در برابر خرید عادی نسیه؛ (ج) برای مرجع، پرداخت عادی در برابر دریافت عادی (حساب‌ها و طرف‌ها جابه‌جا). برای one-hot فاصله در بلوک حساب + طرف notebook 02 گرفته می‌شود و برای embeddingها فاصلهٔ اقلیدسی √(2 − 2·cos). فرضیه (D16): ‏one-hot جفت (الف) و (ب) را به یک فاصله می‌گذارد چون در هر دو یک حساب در یک slot عوض شده؛ نام‌ها (الف) را بسیار نزدیک‌تر از (ب) می‌کنند چون «Bank» و «Bank (Postbank)» تقریباً یکسان خوانده می‌شوند — و این ممکن است M7 را از دید یک detector مبتنی بر فاصله پنهان کند. این فقط توصیف نمایش است؛ تصمیم با detectorهای notebook 05 است.

</div>

In [6]:
# cell_04_06
X, feature_info, params = fe.load_encoding(DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json", meta=meta)
acct_side = [i for i, f in enumerate(feature_info)
             if f["slot"] is not None and ("_acct_" in f["name"] or "_cd_" in f["name"])]   # the 116 columns
tc = sentences["tc"].to_numpy()

pairs = {
    "(a) M7: payment via Bank (1200) vs via Bank (Postbank)":
        ("Debit Trade payables; Credit Bank (Postbank).", "Debit Trade payables; Credit Bank."),
    "(b) M1: purchase paid from cash-in-hand vs on account":
        ("Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Trade payables.",
         "Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Cash-in-hand."),
    "(c) reference: payment vs receipt":
        ("Debit Trade payables; Credit Bank (Postbank).", "Debit Bank (Postbank); Credit Trade receivables."),
}
rows = {}
for name, (a, b) in pairs.items():
    ia, ib = int(np.flatnonzero(tc == a)[0]), int(np.flatnonzero(tc == b)[0])   # one entry of each pattern
    row = {"one-hot (account + side block)": float(np.linalg.norm((X[ia, acct_side] - X[ib, acct_side]).toarray()))}
    for t in embed.TEMPLATES:
        cos = float(emb[t][index[t][ia]] @ emb[t][index[t][ib]])
        row[f"{t}: cosine"] = cos
        row[f"{t}: distance"] = float(np.sqrt(max(0.0, 2 - 2 * cos)))
    rows[name] = row
pd.DataFrame(rows).T.round(3)

,one-hot (account + side block),tc: cosine,tc: distance,bak: cosine,bak: distance
(a) M7: payment via Bank (1200) vs via Bank (Postbank),1.414,0.950,0.316,0.986,0.165
(b) M1: purchase paid from cash-in-hand vs on account,1.414,0.972,0.238,0.989,0.145
(c) reference: payment vs receipt,2.000,0.892,0.465,0.982,0.190


**Result** (run of 27 Nov).
- **One-hot** puts (a) and (b) at the same distance √2 = 1.414 (one account replaced in one slot = two columns differ), and the reference pair (c) at 2.0 (the accounts of both slots differ).
- **`tc` (names):** cosine 0.950 for (a), 0.972 for (b), 0.892 for (c). Against the hypothesis, the M7 pair (a) is *not* closer than the M1 pair (b): the model sees a whole-sentence difference, and "Cash-in-hand" vs "Trade payables" changes less of the sentence's meaning than expected. The reference pair (c) is the furthest apart, as for one-hot.
- **`bak` (codes):** all three cosines lie between 0.982 and 0.989 — the code template hardly separates the patterns, including the reference pair. This supports using names (D16).
- The embedding distances are much smaller than the one-hot ones (0.15–0.47 against 1.4–2.0); how the embedding block is scaled against the other columns (D17) is therefore decided before any detector is run on it (notebook 05).


<div dir="rtl">

**نتیجه** (اعداد embedding از اجرای 26 نوامبر؛ بعد از اجرای دوباره چک شوند).
- **one-hot** جفت (الف) و (ب) را به یک فاصله، √2 = 1.414، می‌گذارد (یک حساب در یک slot عوض شده = دو ستون فرق دارند) و جفت مرجع (ج) را 2.0 (حساب هر دو slot فرق دارد).
- **`tc` (نام‌ها):** کسینوس 0.950 برای (الف)، 0.972 برای (ب) و 0.892 برای (ج). برخلاف فرضیه، جفت M7 ‏(الف) نزدیک‌تر از جفت M1 ‏(ب) *نیست*: مدل تفاوت کل جمله را می‌بیند و «Cash-in-hand» در برابر «Trade payables» کمتر از انتظار معنای جمله را عوض می‌کند. جفت مرجع (ج)، مثل one-hot، از همه دورتر است.
- **`bak` (کدها):** هر سه کسینوس بین 0.982 و 0.989 است — قالب کددار الگوها را تقریباً از هم جدا نمی‌کند، حتی جفت مرجع را. این استفاده از نام‌ها را تأیید می‌کند (D16).
- فاصله‌های embedding بسیار کوچک‌تر از one-hot است (0.15 تا 0.47 در برابر 1.4 تا 2.0)؛ پس مقیاس بلوک embedding نسبت به بقیهٔ ستون‌ها (D17) باید پیش از اجرای هر detector روی آن تعیین شود (notebook 05).

</div>

## 6. Save

**Goal.** Store per template the unique embeddings (29 × 384), the index of every entry in `meta.parquet` order, and a JSON with the sentences, the number of entries per sentence, the model and the library versions — the versions because a different model release would give different vectors.


<div dir="rtl">

**هدف.** برای هر قالب ذخیره می‌شود: embeddingهای یکتا (29 × 384)، اندیس هر سند به ترتیب `meta.parquet`، و یک JSON با جمله‌ها، تعداد سند هر جمله، نام مدل و نسخهٔ کتابخانه‌ها — نسخه‌ها چون انتشار دیگری از مدل بردارهای دیگری می‌دهد.

</div>

In [7]:
# cell_04_07
import sentence_transformers, torch
info = {"model": embed.MODEL_NAME, "normalised": True, "line_order": "D9",
        "sentence_transformers": sentence_transformers.__version__, "torch": torch.__version__,
        "dim": int(emb["tc"].shape[1]), "n_entries": int(len(meta))}
for t in embed.TEMPLATES:
    embed.save(DERIVED, t, texts[t], index[t], emb[t], info)
    print("saved", DERIVED / f"emb_{t}.npy", "+ _index.npy + .json")

saved C:\MyWorks\Thesis\Jet\data\derived\emb_tc.npy + _index.npy + .json
saved C:\MyWorks\Thesis\Jet\data\derived\emb_bak.npy + _index.npy + .json


**Result.** Three files per template in `data/derived/` (not in git, rebuilt by this notebook).


<div dir="rtl">

**نتیجه.** برای هر قالب سه فایل در `data/derived/` (در git نیستند و این notebook دوباره می‌سازدشان).

</div>

## 7. How later notebooks use it

**Goal.** Show the two pieces notebook 05 combines: the embedding expanded to one row per entry, and the one-hot columns that stay next to it (every column of the chosen setting except the per-slot account and side one-hots, D16).


<div dir="rtl">

**هدف.** نشان دادن دو قطعه‌ای که notebook 05 ترکیب می‌کند: embedding که به یک سطر برای هر سند باز می‌شود، و ستون‌های one-hot که کنارش می‌مانند (همهٔ ستون‌های تنظیم انتخاب‌شده به جز one-hotهای حساب و طرف در هر slot، D16).

</div>

In [8]:
# cell_04_08
E, info_tc = embed.load(DERIVED, "tc")                       # one row per entry, meta order
assert E.shape == (len(meta), 384)
cols = embed.non_account_columns(feature_info, fe.ABLATIONS["main"])
print("T-C embedding per entry:", E.shape, "| distinct rows:", len(info_tc["texts"]))
print(f"non-account one-hot columns kept next to it (setting 'main'): {len(cols)}")
print([feature_info[i]["name"] for i in cols])

T-C embedding per entry: (102429, 384) | distinct rows: 29
non-account one-hot columns kept next to it (setting 'main'): 15
['s1_amount', 's2_amount', 's3_amount', 's4_amount', 'user_Alice', 'user_Bob', 'user_Charlie', 'user_David', 'user_Fred', 'user_Max', 'weekend_weekday', 'weekend_sat', 'weekend_sun', 'nwh', 'lag_days']


**Result.** 102,429 × 384 embedding rows (29 distinct) and 15 one-hot columns next to them: the four slot amounts, six users, the three weekend categories, `nwh` and `lag_days`. Main setting of the embedding encoding = 384 + 15 columns before the weighting of D17 (notebook 05); 131 − 116 = 15, so the two encodings differ only in the account block.

## Notes for the thesis
- Ch. 3 (encoding): the template table, the three changes to Bakumenko et al. (2025) and why (D16); only 29 distinct sentences, so the embedding is a semantic embedding of the booking pattern; numbers stay as columns (Wallace et al., 2019).
- Ch. 3 / 4 (RQ2): only 118 anomalies (all M7, 58 of the M1) create patterns of their own; section 5 gives the first evidence on how the two encodings place them; the detectors in notebook 05 decide.
- D17 must be settled first: the embedding distances are three to ten times smaller than the one-hot ones (0.15–0.47 against 1.4–2.0).


<div dir="rtl">

**نتیجه.** ‏102,429 × 384 سطر embedding (29 سطر متمایز) و 15 ستون one-hot کنارش: چهار مبلغ slot، شش کاربر، سه دستهٔ weekend، ‏`nwh` و `lag_days`. تنظیم اصلی encoding ‏embedding پیش از وزن‌دهی D17 (notebook 05) = ‏384 + 15 ستون؛ چون 131 − 116 = 15، دو encoding فقط در بلوک حساب با هم فرق دارند.

</div>